In [ ]:
import pandas as pd
import numpy as np

# ------------------------- STEP 1: LOAD DATA -------------------------

# Load patient demographics
patients = pd.read_csv("/root/MIMICIV/data/mimic-iv-3.1/hosp/patients.csv")

# Load hospital admissions
admissions = pd.read_csv("/root/MIMICIV/data/mimic-iv-3.1/hosp/admissions.csv")

# Load ICU stays
icustays = pd.read_csv("/root/MIMICIV/data/mimic-iv-3.1/icu/icustays.csv")

# Load diagnoses (ICD codes assigned to patients)
diagnoses = pd.read_csv("/root/MIMICIV/data/mimic-iv-3.1/hosp/diagnoses_icd.csv")

# Load prescriptions (medications administered)
prescriptions = pd.read_csv("/root/MIMICIV/data/mimic-iv-3.1/hosp/prescriptions.csv")

# Load medical procedures (ICD codes for treatments)
procedures = pd.read_csv("/root/MIMICIV/data/mimic-iv-3.1/hosp/procedures_icd.csv")

# Load ICD descriptions (for both ICD-9 and ICD-10)
icd_diagnoses = pd.read_csv("/root/MIMICIV/data/mimic-iv-3.1/hosp/d_icd_diagnoses.csv.gz")
icd_procedures = pd.read_csv("/root/MIMICIV/data/mimic-iv-3.1/hosp/d_icd_procedures.csv.gz")

# Convert ICU stay timestamps to datetime format
icustays['intime'] = pd.to_datetime(icustays['intime'], errors='coerce')
icustays['outtime'] = pd.to_datetime(icustays['outtime'], errors='coerce')


# Define mapping for admission types
emergency_types = ['DIRECT EMER.', 'EW EMER.', 'URGENT']
normal_types = ['AMBULATORY OBSERVATION', 'DIRECT OBSERVATION', 'ELECTIVE', 
                'EU OBSERVATION', 'OBSERVATION ADMIT', 'SURGICAL SAME DAY ADMISSION']

# Create a new column 'admission_category' based on the mapping
admissions['admission_category'] = admissions['admission_type'].apply(
    lambda x: 'EMERGENCY' if x in emergency_types else 'NORMAL'
)

# Check if mapping worked correctly
print(admissions['admission_category'].value_counts())

# ------------------------- STEP 2: MAP ICD CODES TO DESCRIPTIONS -------------------------

# Merge diagnoses with ICD descriptions
diagnoses = diagnoses.merge(icd_diagnoses, on=["icd_code", "icd_version"], how="left")
diagnoses['diagnosis_description'] = diagnoses['long_title'].fillna("Unknown diagnosis")

# Merge procedures with ICD descriptions
procedures = procedures.merge(icd_procedures, on=["icd_code", "icd_version"], how="left")
procedures['procedure_description'] = procedures['long_title'].fillna("Unknown procedure")

# Keep only necessary columns
diagnoses = diagnoses[['subject_id', 'hadm_id', 'diagnosis_description', 'icd_code']]
procedures = procedures[['subject_id', 'hadm_id', 'procedure_description', 'icd_code']]

# ------------------------- STEP 3: COMPUTE AGE AT EVENTS & MORTALITY -------------------------

# Keep only relevant patient information
patients = patients[['subject_id', 'gender', 'anchor_age', 'anchor_year', 'dod']]

# Merge patient data into all event tables
admissions = admissions.merge(patients, on='subject_id', how='left')
icustays = icustays.merge(patients, on='subject_id', how='left')
diagnoses = diagnoses.merge(patients, on='subject_id', how='left')
procedures = procedures.merge(patients, on='subject_id', how='left')
prescriptions = prescriptions.merge(patients, on='subject_id', how='left')

# Convert date columns
admissions['admittime'] = pd.to_datetime(admissions['admittime'])
prescriptions['starttime'] = pd.to_datetime(prescriptions['starttime'])
patients['dod'] = pd.to_datetime(patients['dod'], errors='coerce')

# Compute age at event time
admissions['age_at_event'] = admissions['anchor_age'] + (admissions['admittime'].dt.year - admissions['anchor_year'])
prescriptions['age_at_event'] = prescriptions['anchor_age'] + (prescriptions['starttime'].dt.year - prescriptions['anchor_year'])

# Assign age at death for deceased patients
patients['age_at_death'] = patients['anchor_age'] + (patients['dod'].dt.year - patients['anchor_year'])
patients['death_flag'] = patients['dod'].notna().astype(int)

# Merge mortality data into admissions
admissions = admissions.merge(patients[['subject_id', 'death_flag', 'age_at_death']], on='subject_id', how='left')

##### Adding Died During Visit variable 
# Convert discharge time to datetime format
admissions['dischtime'] = pd.to_datetime(admissions['dischtime'], errors='coerce')

# Determine if the patient died during this admission
admissions['died_during_visit'] = (
    (admissions['death_flag'] == 1) & 
    (admissions['dod'] >= admissions['admittime']) & 
    (admissions['dod'] <= admissions['dischtime'])
).astype(int)

##### Creating how many days until the next hospitalization
# Sort admissions by patient ID and admission time
admissions = admissions.sort_values(by=['subject_id', 'admittime'])

# Get the next admission time per patient
admissions['next_admittime'] = admissions.groupby('subject_id')['admittime'].shift(-1)

# Compute the number of days until the next visit
admissions['days_until_next_visit'] = (admissions['next_admittime'] - admissions['dischtime']).dt.days

# Compute the number of minutes until the next visit
admissions['mins_until_next_visit'] = (admissions['next_admittime'] - admissions['dischtime']).dt.total_seconds() / 60

# Fill NaN values with -1 (for last visit of each patient)
admissions['days_until_next_visit'].fillna(-1, inplace=True)

admissions['mins_until_next_visit'].fillna(-1, inplace=True)

##### How many days from the last visit to the death?
# Get the last discharge time per patient
last_discharge = admissions.groupby('subject_id')['dischtime'].max().reset_index()
last_discharge.rename(columns={'dischtime': 'last_dischtime'}, inplace=True)

# Merge last discharge time into patients data
patients = patients.merge(last_discharge, on='subject_id', how='left')

# Calculate days from last visit to death (if patient died after last visit)
patients['days_from_last_visit_to_death'] = (patients['dod'] - patients['last_dischtime']).dt.days

# If patient didn't die, set the value to NaN or -1
patients.loc[patients['death_flag'] == 0, 'days_from_last_visit_to_death'] = np.nan

admissions = admissions.merge(
    patients[['subject_id', 'days_from_last_visit_to_death']], 
    on='subject_id', 
    how='left'
)

# ------------------------- STEP 4: BUILD TABULAR DATASET -------------------------

# Select main features
features = admissions[['subject_id',
                       'hadm_id',
                       'admittime',
                       'dischtime',
                       'age_at_event',
                       'gender',
                       'admission_type',
                       'admission_category',
                       'discharge_location',
                       'death_flag',
                       'age_at_death',
                       'died_during_visit',
                       'days_from_last_visit_to_death',
                       'days_until_next_visit',
                       'mins_until_next_visit'
                       ]]

# ICU stays: Number of ICU admissions and length of stay per hospitalization
icu_summary = icustays.groupby('hadm_id').agg(
    icu_admissions=('stay_id', 'count'),
    icu_days=('intime', lambda x: (x.max() - x.min()).days)
).reset_index()

# Diagnoses: Count number of diagnoses per hospitalization
diagnosis_summary = diagnoses.groupby('hadm_id').agg(
    num_diagnoses=('icd_code', 'count'),
    diagnosis_list=('diagnosis_description', lambda x: list(x.unique()))
).reset_index()

# Procedures: Count number of procedures per hospitalization
procedure_summary = procedures.groupby('hadm_id').agg(
    num_procedures=('icd_code', 'count'),
    procedure_list=('procedure_description', lambda x: list(x.unique()))
).reset_index()

# Medications: Count number of prescribed drugs per hospitalization
med_summary = prescriptions.groupby('hadm_id').agg(
    num_medications=('drug', 'count'),
    medication_list=('drug', lambda x: list(x.unique()))
).reset_index()

# Merge all features into a single dataset
dataset = features.merge(icu_summary, on='hadm_id', how='left')
dataset = dataset.merge(diagnosis_summary, on='hadm_id', how='left')
dataset = dataset.merge(procedure_summary, on='hadm_id', how='left')
dataset = dataset.merge(med_summary, on='hadm_id', how='left')

# Fill missing values with 0
dataset.fillna({'icu_admissions': 0, 'icu_days': 0, 'num_diagnoses': 0, 'num_procedures': 0, 'num_medications': 0}, inplace=True)
dataset.fillna({'diagnosis_list': 'None', 'procedure_list': 'None', 'medication_list': 'None'}, inplace=True)

In [ ]:
admissions['subject_id'].nunique()

In [ ]:
dataset[dataset['subject_id']==11530780][['subject_id','admittime', 'dischtime', 'days_until_next_visit']]

In [ ]:
admissions_sorted = dataset.sort_values(['subject_id', 'admittime'])

In [ ]:
# Check for overlaps (current visit starts before previous ends)
admissions_sorted['overlaps'] = (
    admissions_sorted.groupby('subject_id')['admittime'].shift(-1) < admissions_sorted['dischtime']
)

In [ ]:
admissions_sorted[admissions_sorted['overlaps']==True]

In [ ]:
TEST_df = admissions_sorted[admissions_sorted['subject_id']==10076639]

In [ ]:
TEST_df

In [ ]:
TEST_df = TEST_df.sort_values(by=['subject_id', 'admittime'])
TEST_df = TEST_df.reset_index(drop=True)

In [ ]:
TEST_df

In [ ]:
admissions_sorted['subject_id'].nunique()


What can be done in this situation? We can try different approaches, starting from the concatenation in an unique visit of the overlapping visits for all 48 patients. Or, since there are just 48 patients with overlapping over 223452 (i.e., 0.02%) we decide to remove those patients from our dataset.

In [ ]:
dataframe_no_overlap = admissions_sorted.groupby('subject_id').filter(lambda x: all(x['overlaps']==False))

In [ ]:
dataframe_no_overlap['subject_id'].nunique() # no more overlaps

In [ ]:
dataframe_no_overlap[dataframe_no_overlap['mins_until_next_visit']==0]


In [ ]:
dataframe_no_overlap[dataframe_no_overlap['subject_id']==10002930][['subject_id', 'admittime', 'dischtime', 'days_until_next_visit', 'mins_until_next_visit']]

In [ ]:
# Let us load the landmark_df_evo dataset
landmark_df_evo = pd.read_csv("/root/MIMICIV/src/landmark_df_evo.csv")

In [ ]:
import pandas as pd

In [ ]:
landmark_df_evo_correct = pd.read_csv("/root/MIMICIV/src/landmark_df_evo_correct.csv")

In [ ]:
landmark_df_evo['subject_id'].nunique()

In [ ]:
landmark_df_evo_correct['subject_id'].nunique()

In [ ]:
#landmark_df_evo.head()

In [ ]:
dataframe_no_overlap_2_merge = dataframe_no_overlap[['subject_id', 'hadm_id', 'admittime', 'dischtime', 'days_until_next_visit', 'mins_until_next_visit']].copy()
dataframe_no_overlap_2_merge.head()

In [ ]:
dataframe_no_overlap_2_merge['subject_id'].nunique()

In [ ]:
# Ensure your DataFrame is sorted by patient and landmark_visit
dataframe_no_overlap_2_merge = dataframe_no_overlap_2_merge.sort_values(['subject_id', 'admittime']).reset_index(drop=True)

# Function to perform transformation
def transform_to_past_variable(group):
    group = group.sort_values('admittime').copy()
    # Shift the values down (future to past)
    group['days_since_last_visit'] = group['days_until_next_visit'].shift(1)
    # First landmark always has -1
    group['days_since_last_visit'].iloc[0] = -1
    # same reasoning for minutes
    group['mins_since_last_visit'] = group['mins_until_next_visit'].shift(1)
    # First landmark always has -1
    group['mins_since_last_visit'].iloc[0] = -1

    return group

# Apply the transformation per patient
dataframe_no_overlap_2_merge = dataframe_no_overlap_2_merge.groupby('subject_id').apply(transform_to_past_variable).reset_index(drop=True)


In [ ]:
dataframe_no_overlap_2_merge.head()


In [ ]:
dataframe_no_overlap_2_merge.to_csv("/root/MIMICIV/src/dataframe_no_overlap_2_merge.csv", index=False)

In [ ]:
import pandas as pd

In [ ]:
dataframe_no_overlap_2_merge = pd.read_csv("/root/MIMICIV/src/dataframe_no_overlap_2_merge.csv")
landmark_df_evo_correct = pd.read_csv("/root/MIMICIV/src/landmark_df_evo_correct.csv")


In [ ]:
landmark_df_evo['subject_id'].nunique()  # Check unique patients in landmark dataset<

Landmark_df_evo (the dataset on which we are basing all the analysis) looses 55_037 patients from admissions (after removong the 48 patients). 
We need to investigate starting from the dataset at the base of the landmark_evo dataset what happens.

In [ ]:
mimic_tab_death_visit_evo = pd.read_csv("mimiciv_clinical_dataset_tabular_death_visit_evo.csv")


In [ ]:
mimic_tab_death_visit_evo['subject_id'].nunique()

Dataset mimiciv_clinical_dataset_tabular_death_visit_evo.csv has the right number of patients. Then that's good. So, something happend in creating landmark_evo dataset.
Please, refer to the code **check_data_creation_evo.ipynb** for this.

We produced the **landmark_df_evo_correct** which has exactly the same amount of patients of the admissions (it could have been a problem in saving).

In [ ]:
landmark_df_evo_correct_no_overlap = landmark_df_evo_correct.merge(dataframe_no_overlap_2_merge, on=['subject_id', 'hadm_id'], how='inner')

In [ ]:
landmark_df_evo_correct_no_overlap

In [ ]:
landmark_df_evo_correct_no_overlap['subject_id'].nunique()

Now let's test whether the full prompting scheme works here. There is a problem: few patients are readmitted almost at the same time of dischargment of the previous visit. What to do? The easy approach is to force also for them 1 day-delay (in this case 1 day will be the smallest time unit). Another possible approach would be to go to minutes/seconds in such cases. Howevere there are also cases where the patient gets admitted exactly at the same time of discharge. 

In [ ]:
landmark_df_evo_correct_no_overlap['days_since_last_visit'] = landmark_df_evo_correct_no_overlap['days_since_last_visit'].replace(0, 1)

In [ ]:
import numpy as np

In [ ]:
landmark_df_evo_correct_no_overlap['days_since_last_visit_cumulate'] = landmark_df_evo_correct_no_overlap.groupby('subject_id')['days_since_last_visit'].transform(lambda x: [list(x[:i+1]) for i in range(len(x))])
landmark_df_evo_correct_no_overlap['days_since_last_visit_cumulate_sum'] = landmark_df_evo_correct_no_overlap['days_since_last_visit_cumulate'].apply(lambda x: np.cumsum([y for y in x if y > 0][::-1])[::-1] if isinstance(x, list) else [-1])



In [ ]:
landmark_df_evo_correct_no_overlap

In [ ]:
visit_counts = landmark_df_evo_correct_no_overlap['subject_id'].value_counts()
selected_patients = visit_counts[visit_counts == 3].index
landmark_df_evo_correct_no_overlap_selected = landmark_df_evo_correct_no_overlap[landmark_df_evo_correct_no_overlap['subject_id'].isin(selected_patients)].copy()

In [ ]:
landmark_df_evo_correct_no_overlap_selected

In [ ]:
import ast
import random
from collections import Counter

def full_narrative(row):
    narrative = f"You are a Doctor.\nWhat is the probability of death in the next 90 days from today for this {row['age_at_landmark']}-year-old {row['gender']} patient?\n"
    current_visit = row['landmark_visit']
    narrative += f"Today is the {current_visit} visit.\n"

    max_visit = int(row['landmark_visit'])

    # if row['days_since_previous_visit'] != -1:
    #     narrative += f"Last visit happened {row['days_since_previous_visit']} days ago."

    narrative += "\nDiagnosis history:"
    for past_visit, diags in reversed(list(ast.literal_eval(row['diag_per_visit']).items())):
        if past_visit == max_visit:
            narrative += f"\nToday: {'; '.join(diags)}."
        else:
            narrative += f"\n{int(row['days_since_last_visit_cumulate_sum'][int(past_visit) -1])} days ago: {'; '.join(diags)}."

    narrative += "\nPrescriptions history:"
    for past_visit, meds in reversed(ast.literal_eval(row['meds_per_visit']).items()):
        if past_visit ==  max_visit:
            narrative += f"\nToday: {'; '.join(meds)}."
        else:
            narrative += f"\n{int(row['days_since_last_visit_cumulate_sum'][past_visit-1])} days ago: {'; '.join(meds)}."

    narrative += "\nProcedures history:"
    for past_visit, proc in reversed(ast.literal_eval(row['proc_per_visit']).items()):
        if past_visit == max_visit:
            narrative += f"\nToday: {'; '.join(proc)}."
        else:
            narrative += f"\n{int(row['days_since_last_visit_cumulate_sum'][past_visit-1])} days ago: {'; '.join(proc)}."
    
    return narrative

def full_narrative_no_time_rnd(row):
    narrative = f"You are a Doctor.\nWhat is the probability of death in the next 90 days from today for this {row['age_at_landmark']}-year-old {row['gender']} patient?\n"

    # Diagnosi
    all_diags = []
    diag_dict = ast.literal_eval(row['diag_per_visit'])
    for diags in diag_dict.values():
        all_diags.extend(diags)
    random.shuffle(all_diags)
    narrative += "\nDiagnosis history:\n" + '; '.join(all_diags) + "."

    # Prescrizioni
    all_meds = []
    meds_dict = ast.literal_eval(row['meds_per_visit'])
    for meds in meds_dict.values():
        all_meds.extend(meds)
    random.shuffle(all_meds)
    narrative += "\nPrescriptions history:\n" + '; '.join(all_meds) + "."

    # Procedure
    all_procs = []
    proc_dict = ast.literal_eval(row['proc_per_visit'])
    for procs in proc_dict.values():
        all_procs.extend(procs)
    random.shuffle(all_procs)
    narrative += "\nProcedures history:\n" + '; '.join(all_procs) + "."

    return narrative

def no_narrative_prompt(row):
    narrative = "Variables: "
    if pd.notna(row['diag_text']) and row['diag_text'].strip():
        narrative += f" {row['diag_text']}"
    if pd.notna(row['med_text']) and row['med_text'].strip():
        narrative += f" {row['med_text']}"
    if pd.notna(row['proc_text']) and row['proc_text'].strip():
        narrative += f" {row['proc_text']}"

    return narrative

def compact_narrative_prompt(row):
    narrative = f"You are a Doctor.\nWhat is the probability of death in the next 90 days for {row['age_at_landmark']}-year-old {row['gender']} patient?\n"
    current_visit = row['landmark_visit']
    type = row['admission_category']
    narrative += f"Visit number {current_visit} - {type} \n"

    if row['days_since_previous_visit'] != -1:
        narrative += f"Last visit happened {row['days_since_previous_visit']} days ago."

    # --- Diagnosi ---
    narrative += "\nDIAGNOSIS HISTORY:"
    diag_per_visit = ast.literal_eval(row['diag_per_visit'])
    
    # Conta frequenze
    all_diags = []
    for diags in diag_per_visit.values():
        all_diags.extend(diags)
    diag_counts = Counter(all_diags)

    # Diagnosi croniche (almeno 2 visite)
    chronic_diags = [d for d, c in diag_counts.items() if c >= 2]

    # Diagnosi nuove solo in questa visita
    current_diags = diag_per_visit[int(current_visit)]
    new_diags = [d for d in current_diags if diag_counts[d] == 1]

    if chronic_diags:
        narrative += f"\nChronic diagnoses: {'; '.join(chronic_diags)}."
    if new_diags:
        narrative += f"\nNew diagnoses in this visit: {'; '.join(new_diags)}."
    if not chronic_diags and not new_diags:
        narrative += "\nNo diagnoses recorded."

    # --- Farmaci ---
    narrative += "\nPRESCRIPTIONS HISTORY:"
    meds_per_visit = ast.literal_eval(row['meds_per_visit'])
    
    all_meds = []
    for meds in meds_per_visit.values():
        all_meds.extend(meds)
    med_counts = Counter(all_meds)

    chronic_meds = [m for m, c in med_counts.items() if c >= 2]
    current_meds = meds_per_visit[int(current_visit)]
    new_meds = [m for m in current_meds if med_counts[m] == 1]

    if chronic_meds:
        narrative += f"\nChronic medications: {'; '.join(chronic_meds)}."
    if new_meds:
        narrative += f"\nNew medications in this visit: {'; '.join(new_meds)}."
    if not chronic_meds and not new_meds:
        narrative += "\nNo medications recorded."

    # --- Procedure ---
    narrative += "\nPROCEDURES HISTORY:"
    proc_per_visit = ast.literal_eval(row['proc_per_visit'])

    all_proc = []
    for procs in proc_per_visit.values():
        all_proc.extend(procs)
    proc_counts = Counter(all_proc)

    chronic_proc = [p for p, c in proc_counts.items() if c >= 2]
    current_proc = proc_per_visit[int(current_visit)]
    new_proc = [p for p in current_proc if proc_counts[p] == 1]

    if chronic_proc:
        narrative += f"\nChronic procedures: {'; '.join(chronic_proc)}."
    if new_proc:
        narrative += f"\nNew procedures in this visit: {'; '.join(new_proc)}."
    if not chronic_proc and not new_proc:
        narrative += "\nNo procedures recorded."

    return narrative


In [ ]:
train_df = landmark_df_evo_correct_no_overlap_selected[landmark_df_evo_correct_no_overlap_selected['landmark_visit'] == 3].copy()

In [ ]:
train_df.columns

In [ ]:
problem_subjects = train_df[train_df.apply(lambda x: len(x['days_since_last_visit_cumulate_sum'])==0, axis=1)]['subject_id'].reset_index(drop=True).tolist()

In [ ]:
landmark_df_evo_correct_no_overlap_selected[landmark_df_evo_correct_no_overlap_selected['subject_id'].isin(problem_subjects)]

In [ ]:
train_texts_full = train_df.apply(full_narrative, axis=1).tolist()
train_texts_full_no_time_rnd = train_df.apply(full_narrative_no_time_rnd, axis=1).tolist()
train_texts_no_narrative = train_df.apply(no_narrative_prompt, axis = 1).tolist()
train_texts_compact = train_df.apply(compact_narrative_prompt, axis = 1).tolist()

In [ ]:
print(train_texts_full[12])

In [ ]:
print(train_texts_full_no_time_rnd[12])

In [ ]:
print(train_texts_no_narrative[12])

In [ ]:
print(train_texts_compact[12])

In [ ]:
train_df.iloc[12]

In [ ]:
#ast.literal_eval(train_df['diag_text'])
row = train_df.iloc[12]
list(ast.literal_eval(row['diag_per_visit']).items())

In [ ]:
# Let's see lenghts:
len_full = len(train_texts_full[12])
len_full_no_time_rnd = len(train_texts_full_no_time_rnd[12])
len_no_narrative = len(train_texts_no_narrative[12])
len_compact = len(train_texts_compact[12])
print(len_full, len_full_no_time_rnd, len_no_narrative, len_compact)
